# Agentic-checkout (`linkagnt*`) coverage across card panels

**Goal.** Find which card / bank panels can see Stripe Link agentic-checkout transactions, de-duplicate overlapping sources, and track daily volume around the Muse launch (2026-09-08).

**Steps**
1. Source inventory (279 card tables, Yodlee 201 feeds, Fable EU panels).
2. Presence check on 2026-09-15.
3. Duplicate detection across sources.
4. Daily / weekly series of agentic transactions vs panel volume.
5. Top agentic merchants.

**Agentic descriptor.** Stripe renamed the descriptor on 2026-09-23/24: `LINKAGNT* <merchant> WWW.LINK.COM CA` became `LINK* <merchant> WWW.LINK.COM CA`. Both forms are matched (`AGENTIC_SQL`). Bare `LINK*` must be anchored on `www.link.co` / `south san fra` (Stripe's city), otherwise it picks up `CASH APP*... LINK* Oakland` and `PTO LINK*` noise. `LINK.COM* <merchant>` is the ordinary (non-agent) Link wallet and is excluded.

## Status as of 2026-10-07 (run via SRS MCP Snowflake; local JWT login broken)

**1. Inventory**
- The three 279 databases (`SHARE_CE_SPEND_279_FINANCIAL_PRIME_ENRICHED`, `..._TAU`, `SHARE_EARNEST_279`) are **exact mirrors**: identical row counts for every table, and identical `HASH_AGG(partner_transaction_id, merchant, amount)` for `PRIME` and `PRIME_TAU` on 2026-09-15.
- Only **`PRIME`** and **`PRIME_TAU`** (plus their monthly `*_SNAPSHOTS` copies) are live. Every `DIRECT*` and `*SIGMA` table was last altered in 2025 and has no 2026-09 rows.
- Yodlee: feeds 1 and 2 stopped on 2025-09-15. **Feeds 3, 4 and 6** (card + bank) are live.
- MCP role cannot read `PRIME_TAU` / `DIRECT_*` in the non-TAU CE database; use the `_TAU` database or `SHARE_EARNEST_279`. `*_SNAPSHOTS` tables time out even on one day (no pruning) and were not tested.

**2. Presence on 2026-09-15** (distinct transactions)

| Source | Day txns | `linkagnt` txns | Note |
|---|---:|---:|---|
| 279 `PRIME` | 7,425,734 | 9 | 15 rows: each txn tagged twice (`link_agent_wallet` + real merchant) |
| 279 `PRIME_TAU` | 12,512,717 | 3 | no `link_agent_wallet` tag |
| Yodlee F3 card | 1,136,526 | 2 | |
| Yodlee F4 card | 733,155 | 7 | |
| Yodlee F6 card | 418,853 | 2 | |
| Yodlee F3 bank | 22,143,312 | 8 | debit-card purchases |
| Yodlee F4 bank | 949,378 | 0 | zero across Aug–Sep |
| Yodlee F6 bank | n/a (count timed out) | 2 | |
| Fable UK/FR/DE/ES/IT | 0.4–1.9M each | 0 | only `link.com*` (regular Link wallet) |

**3. Duplicates**
- `PRIME` vs `PRIME_TAU`: **disjoint** (0 of 1,986 sampled `PRIME_TAU` ids found in `PRIME`; `PRIME_TAU` agentic members absent from `PRIME`). Additive.
- **`PRIME` ⊂ Yodlee.** Yodlee `unique_mem_id` = 279 `member_id`. All 9 `PRIME` agentic txns on 09-15 appear in feeds 3/4/6 (same member, description, amount); Yodlee has 10 more (21 total incl. F6 bank). `PRIME` is a filtered subset of feeds 3/4/6.
- `PRIME_TAU`'s 3 txns (eBay, Topgolf, Domino's) are **not** in feeds 3/4/6, so `PRIME_TAU` is a separate panel.
- **Deduplicated coverage = Yodlee feeds 3/4/6 (card + bank) ∪ 279 `PRIME_TAU`.** On 09-15: 21 + 3 = 24 distinct agentic txns.

**4. Weekly agentic transactions (Yodlee, tightened definition; week starting Monday)**

| Week | F3 card | F4 card | F6 card | F3 bank | F6 bank | Total | per M card txns |
|---|---:|---:|---:|---:|---:|---:|---:|
| 08-03 | 0 | 4 | 0 | 0 | 0 | 4 | 0.2 |
| 08-10 | 0 | 12 | 0 | 0 | 0 | 12 | 0.7 |
| 08-17 | 0 | 0 | 0 | 0 | 0 | 0 | 0.0 |
| 08-24 | 1 | 9 | 6 | 1 | 0 | 17 | 1.0 |
| 08-31 | 1 | 11 | 0 | 4 | 0 | 16 | 0.7 |
| **09-07** (launch) | 10 | 52 | 6 | 15 | 6 | 89 | 4.6 |
| 09-14 | 7 | 58 | 17 | 54 | 17 | 153 | 5.6 |
| 09-21 | 26 | 111 | 28 | 107 | 23 | 295 | 11.2 |
| 09-28 (lagged) | 24 | 96 | 21 | 147 | 29 | 317 | 13.5 |

Weekly card-panel volume: F3 ≈ 7.5M, F4 ≈ 5.2M, F6 ≈ 3.3M txns; F3 bank ≈ 148M rows. Data after 2026-10-02 is incomplete (reporting lag).

- Daily F4 card `LINKAGNT*` series: ~1/day before 09-08, 7–17/day from 09-09; drops to 0 from 09-24 as `LINK*` takes over (14–26/day).
- 279 `PRIME` spot days (distinct agentic): 09-15 = 9, 09-22 = 19, 09-24 = 12, 09-25 = 22, 09-26 = 15, 09-27 = 6, 09-28 = 34. `merchant = 'link_agent_wallet'` tracks the description match closely (19/19 on 09-22) and already covers the new `LINK*` form. `PRIME_TAU` 09-22 = 6 (0 tagged).
- 279 August: single-day queries time out via MCP (older partitions prune poorly). Run the notebook locally for the 279 baseline.

**5. Top agentic merchants (Yodlee cards, 2026-09-01 to 10-02):** eBay (13), Square sellers (9), American Airlines (6), Target.com (6), FareHarbor (5), Walmart.com, Jura, eHawaii.gov, FasTrak, Progressive (median $952), Thorne, Nordstrom, Teleflora, Turo, Viator, Toast cafés, Alaska Air, Zappos, Omni Hotels, Irish Rail. A mix of travel, retail marketplaces, government fees, and food.

**Conclusions for the app**
- Use **Yodlee feeds 3/4/6 (card + bank)** as the primary agentic source, plus **279 `PRIME_TAU`**. Drop 279 `PRIME` (subset of Yodlee), mirror databases, `DIRECT*`/`SIGMA` (stale), Fable (no agent descriptor), and feeds 1/2 (stale).
- Match both `LINKAGNT*` and anchored `LINK*`; the rename on 09-23/24 would otherwise look like a collapse.
- Agentic volume rose roughly 15× from August to late September, starting the week of the Muse launch.

In [ ]:
import os
import pandas as pd
import snowflake.connector

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

# Uses a named connection from ~/.snowflake/connections.toml (same pattern as srs-evals).
conn = snowflake.connector.connect(
    connection_name=os.getenv("SNOWFLAKE_CONNECTION", "default"),
)

def q(sql: str) -> pd.DataFrame:
    df = conn.cursor().execute(sql).fetch_pandas_all()
    df.columns = [c.lower() for c in df.columns]
    return df

SEPT = pd.date_range("2026-09-01", "2026-09-30")
WINDOW = pd.date_range("2026-08-01", "2026-10-02")  # data after 10-02 is incomplete (reporting lag)
PROBE_DAY = "2026-09-15"


def agentic_sql(col: str) -> str:
    """Stripe Link agentic checkout: LINKAGNT* (until 09-23) or LINK* anchored on Stripe's descriptor (from 09-23)."""
    return (f"({col} ILIKE '%linkagnt*%' OR (REGEXP_LIKE({col}, '(^|.*[^a-z.])link\\\\* .*', 'i') "
            f"AND ({col} ILIKE '%www.link.co%' OR {col} ILIKE '%south san fra%')))")

## 1. Source inventory

List every `SPEND_279_TRANSACTIONS*_V3_0` table across the three 279 databases, plus the Yodlee feed panels and Fable transaction tables, with row count and `last_altered` (stale tables are dropped).

In [ ]:
DB_279 = ["SHARE_CE_SPEND_279_FINANCIAL_PRIME_ENRICHED", "SHARE_CE_SPEND_279_FINANCIAL_PRIME_ENRICHED_TAU", "SHARE_EARNEST_279"]

inv = pd.concat([
    *[q(f"""SELECT table_catalog, table_schema, table_name, row_count, last_altered
           FROM {db}.INFORMATION_SCHEMA.TABLES
           WHERE table_name ILIKE 'SPEND_279_TRANSACTIONS%V3_0'""") for db in DB_279],
    q("""SELECT table_catalog, table_schema, table_name, row_count, last_altered
         FROM SHARE_CE_SPEND_201_PRIME_EX_TAU.INFORMATION_SCHEMA.TABLES
         WHERE table_schema ILIKE 'FEED%PANELS' AND table_name ILIKE '%PANELS_V1_0'"""),
    q("""SELECT table_catalog, table_schema, table_name, row_count, last_altered
         FROM SHARE_FABLE.INFORMATION_SCHEMA.TABLES
         WHERE table_schema = 'FABLE_SIGNAL' AND table_name ILIKE '%TXN'"""),
])
inv["is_log"] = inv.table_name.str.contains("_LOG") | inv.table_schema.str.contains("INCREMENTAL")
inv["live"] = pd.to_datetime(inv.last_altered, utc=True) >= pd.Timestamp("2026-09-30", tz="UTC")
inv.sort_values(["table_catalog", "table_schema", "table_name"])

In [ ]:
# One source per live, non-log table. The three 279 databases are mirrors (see status), so SHARE_EARNEST_279 stands in for all.
# Snapshot tables are monthly copies of PRIME / PRIME_TAU; listed but expensive (no date pruning).
def src_279(schema):
    return dict(name=f"279 {schema}", table=f"SHARE_EARNEST_279.{schema}.SPEND_279_TRANSACTIONS_V3_0",
                desc_col="billing_description", date_col="transaction_date", amt_col="transaction_amount",
                txn_id_col="earnest_transaction_id", merchant_col="merchant")

def src_yodlee(feed, kind):
    return dict(name=f"Yodlee F{feed} {kind.lower()}",
                table=f"SHARE_CE_SPEND_201_PRIME_EX_TAU.FEED{feed}_PANELS.FEED{feed}_{kind}_PANELS_V1_0",
                desc_col="description", date_col="transaction_date", amt_col="amount",
                txn_id_col=f"unique_{kind.lower()}_transaction_id", merchant_col=None)

def src_fable(t):
    return dict(name=f"Fable {t}", table=f"SHARE_FABLE.FABLE_SIGNAL.{t}", desc_col="description",
                date_col="txndate", amt_col="spendout", txn_id_col="fdtxnkey", merchant_col=None)

SOURCES = [
    src_279("PRIME"), src_279("PRIME_TAU"),
    *[src_yodlee(f, k) for f in (3, 4, 6) for k in ("CARD", "BANK")],
    *[src_fable(t) for t in ("UK_V2_TXN", "FR_V2_TXN", "FR_V3_TXN", "DE_V2_TXN", "ES_V2_TXN", "IT_V2_TXN")],
]
SNAPSHOT_SOURCES = [src_279("PRIME_SNAPSHOTS"), src_279("PRIME_SNAPSHOTS_TAU")]
# Stale (last altered 2025, no 2026-09 rows): 279 DIRECT*, *SIGMA; Yodlee feeds 1 and 2.
[s["name"] for s in SOURCES]

## 2. Presence check on 2026-09-15

Total distinct transactions that day, and distinct agentic transactions (`linkagnt*` / anchored `link*`). `hit_rows` > `hit_txn` means the source repeats a transaction per merchant tag.

In [ ]:
def scan_daily(src, where_sql, days, extra_cols=""):
    """One query per day (month-long ILIKE scans time out on the 279 tables). Returns day, total_txn, hit_rows, hit_txn."""
    out = []
    for d in days:
        d = pd.Timestamp(d).date().isoformat()
        df = q(f"""
            SELECT '{d}'::DATE AS day,
                   COUNT(DISTINCT {src['txn_id_col']}) AS total_txn,
                   COUNT_IF({where_sql}) AS hit_rows,
                   COUNT(DISTINCT IFF({where_sql}, {src['txn_id_col']}, NULL)) AS hit_txn {extra_cols}
            FROM {src['table']} WHERE {src['date_col']} = '{d}'
        """)
        out.append(df.assign(source=src["name"]))
    return pd.concat(out, ignore_index=True)

presence = pd.concat([scan_daily(s, agentic_sql(s["desc_col"]), [PROBE_DAY]) for s in SOURCES], ignore_index=True)
presence["present"] = presence.hit_txn > 0
presence[["source", "day", "total_txn", "hit_rows", "hit_txn", "present"]]

## 3. Duplicate detection

- Mirror check: `HASH_AGG` of a full day in each 279 database.
- `PRIME` vs `PRIME_TAU`: id overlap on the probe day.
- 279 vs Yodlee: ids differ, but `member_id` = `unique_mem_id`, so match on (member, amount, normalised description).

In [ ]:
# Mirror check across the three 279 databases
mirror = pd.concat([
    q(f"""SELECT '{db}' AS db, '{sch}' AS sch,
                 HASH_AGG(partner_transaction_id, merchant, transaction_amount) AS h, COUNT(*) AS n
          FROM {db}.{sch}.SPEND_279_TRANSACTIONS_V3_0 WHERE transaction_date = '{PROBE_DAY}'""")
    for sch, dbs in {"PRIME": [DB_279[0], DB_279[2]], "PRIME_TAU": [DB_279[1], DB_279[2]]}.items() for db in dbs
])
mirror

In [ ]:
# Agentic rows on the probe day from every source with hits, keyed on (member, amount, merchant text after the descriptor)
hit_sources = [s for s in SOURCES if s["name"] in set(presence.loc[presence.present, "source"])]
MEMBER = {"billing_description": "member_id", "description": "unique_mem_id"}

rows = []
for s in hit_sources:
    if s["name"].startswith("Fable"):
        continue
    rows.append(q(f"""
        SELECT DISTINCT '{s['name']}' AS source, {MEMBER[s['desc_col']]} AS member, {s['txn_id_col']} AS txn_id,
               ROUND({s['amt_col']}, 2) AS amount, LOWER({s['desc_col']}) AS description
        FROM {s['table']}
        WHERE {s['date_col']} = '{PROBE_DAY}' AND {agentic_sql(s['desc_col'])}
    """))
hits = pd.concat(rows, ignore_index=True)
hits["key"] = hits.member.astype(str) + "|" + hits.amount.astype(str)

# Which sources share each (member, amount) key
overlap = hits.groupby("key").source.apply(lambda s: " + ".join(sorted(set(s)))).value_counts()
print(f"{hits.key.nunique()} distinct agentic transactions on {PROBE_DAY} across {len(hit_sources)} sources")
overlap

In [ ]:
# PRIME vs PRIME_TAU id overlap (sample of PRIME_TAU ids)
q(f"""
    WITH t AS (SELECT partner_transaction_id FROM SHARE_EARNEST_279.PRIME_TAU.SPEND_279_TRANSACTIONS_V3_0
               WHERE transaction_date = '{PROBE_DAY}' LIMIT 2000)
    SELECT COUNT(DISTINCT t.partner_transaction_id) AS sampled, COUNT(DISTINCT p.partner_transaction_id) AS found_in_prime
    FROM t LEFT JOIN (SELECT partner_transaction_id FROM SHARE_EARNEST_279.PRIME.SPEND_279_TRANSACTIONS_V3_0
                      WHERE transaction_date = '{PROBE_DAY}') p USING (partner_transaction_id)
""")

In [ ]:
# Deduplicated set used downstream: Yodlee feeds 3/4/6 (card + bank) + 279 PRIME_TAU.
# 279 PRIME is a subset of the Yodlee feeds (same member ids); mirrors and snapshots are copies.
DEDUP_SOURCES = [s for s in SOURCES if s["name"].startswith("Yodlee") or s["name"] == "279 PRIME_TAU"]
[s["name"] for s in DEDUP_SOURCES]

## 4. Daily series, 2026-08-01 to 2026-10-02

Distinct agentic transactions per day per source, plus total daily transactions, giving agentic share per million. The 279 sources run one query per day; the Yodlee feeds are small enough for one query per source.

In [ ]:
def scan_range(src, where_sql, start, end):
    """Single grouped query; fine for Yodlee card feeds. Falls back to scan_daily for the 279 tables."""
    return q(f"""
        SELECT {src['date_col']}::DATE AS day,
               COUNT(DISTINCT {src['txn_id_col']}) AS total_txn,
               COUNT_IF({where_sql}) AS hit_rows,
               COUNT(DISTINCT IFF({where_sql}, {src['txn_id_col']}, NULL)) AS hit_txn
        FROM {src['table']}
        WHERE {src['date_col']} BETWEEN '{start}' AND '{end}'
        GROUP BY 1
    """).assign(source=src["name"])

series = []
for s in DEDUP_SOURCES + [src_279("PRIME")]:
    w = agentic_sql(s["desc_col"])
    if s["name"].startswith("279"):
        series.append(scan_daily(s, w, WINDOW))
    else:
        series.append(scan_range(s, w, WINDOW[0].date(), WINDOW[-1].date()))
series = pd.concat(series, ignore_index=True)
series["day"] = pd.to_datetime(series.day)
series["per_million"] = series.hit_txn / series.total_txn * 1e6

daily = series.pivot_table(index="day", columns="source", values="hit_txn", aggfunc="sum").fillna(0).astype(int)
daily["dedup_total"] = daily[[s["name"] for s in DEDUP_SOURCES if s["name"] in daily]].sum(axis=1)
daily

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
daily.drop(columns="dedup_total").plot(ax=axes[0], title="Agentic (Link) transactions per day by source")
axes[0].axvline(pd.Timestamp("2026-09-08"), color="grey", ls="--", lw=1)
axes[0].axvline(pd.Timestamp("2026-09-23"), color="grey", ls=":", lw=1)
axes[0].text(pd.Timestamp("2026-09-08"), axes[0].get_ylim()[1] * 0.9, " Muse launch", fontsize=8)
axes[0].text(pd.Timestamp("2026-09-23"), axes[0].get_ylim()[1] * 0.8, " LINKAGNT* to LINK*", fontsize=8)

share = series.pivot_table(index="day", columns="source", values="per_million")
share.rolling(7, min_periods=1).mean().plot(ax=axes[1], title="Agentic txns per million panel txns (7-day mean)")
axes[1].axvline(pd.Timestamp("2026-09-08"), color="grey", ls="--", lw=1)
plt.tight_layout()

In [ ]:
# Spot-check: 279 PRIME raw rows / distinct txns = number of merchant tags per txn (expect ~2 when tagged)
prime = series[series.source == "279 PRIME"]
prime.assign(rows_per_txn=prime.hit_rows / prime.hit_txn.replace(0, pd.NA))[["day", "hit_rows", "hit_txn", "rows_per_txn"]].tail(20)

## 5. Top agentic merchants

Merchant text is the token after `LINKAGNT*` / `LINK*` (skipping `SP `, `DD *`, `QDI*` processor prefixes).

In [ ]:
card_feeds = [s for s in DEDUP_SOURCES if s["name"].startswith("Yodlee") and s["name"].endswith("card")]
union = "\nUNION ALL\n".join(
    f"SELECT '{s['name']}' AS source, description AS x, amount FROM {s['table']} "
    f"WHERE transaction_date BETWEEN '2026-09-01' AND '{WINDOW[-1].date()}' AND description ILIKE '%link%'"
    for s in card_feeds
)
top_merchants = q(f"""
    WITH u AS ({union}),
    a AS (SELECT source, x, amount,
                 TRIM(REGEXP_SUBSTR(UPPER(x), '(LINKAGNT|LINK)\\\\* ?(SP |DD \\\\*|QDI\\\\*)?([A-Z0-9.'' -]{{2,12}})', 1, 1, 'e', 3)) AS merchant
          FROM u WHERE {agentic_sql('x')})
    SELECT merchant, COUNT(*) AS n, ROUND(MEDIAN(amount), 2) AS median_amount, ANY_VALUE(x) AS example
    FROM a GROUP BY 1 ORDER BY 2 DESC LIMIT 40
""")
top_merchants